In [52]:
from transformers import BertTokenizerFast, AutoModelForCausalLM
import torch
import numpy as np

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
model = AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese")

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 50954.04it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ckiplab/gpt2-base-chinese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
text = ["這對夫妻攜手走過四十年，現在依然非常",
        "他們依偎彼此手牽手漫步在沙灘上，看來非常"]

inputs = tokenizer(text,
                   truncation=True,
                   padding=True,
                   return_tensors="pt",
                   return_offsets_mapping=True, max_length=256)
inputs = {k: v.to(device) for k, v in inputs.items()}
ids = inputs["input_ids"]      # BERT token IDs: 0 = [PAD]; 101 = [CLS]; 102 = [SEP]
tokens = [tokenizer.convert_ids_to_tokens(i) for i in ids]
offsets = inputs["offset_mapping"]
inputs.pop("offset_mapping") # remove offset_mapping from inputs before feeding into model

with torch.inference_mode():
    outputs = model(**inputs,
                    output_hidden_states=False)
logits = outputs.logits    # shape = (batch_size, seq_len, vocab_size)
logprobs = torch.log_softmax(logits, dim=-1)
logprobs = logprobs.detach().cpu().numpy()

indices = ids[:, 1:]
token_logprobs = np.array([[logprobs[i, j, idx] for j, idx in enumerate(indices[i])]
                                       for i in range(indices.size(0))])
token_logprobs = np.concatenate((np.zeros((indices.size(0), 1)),
                                 token_logprobs),
                                 axis=1)

print(token_logprobs)

[[  0.          -6.02565956  -4.76508236  -5.11125278  -1.28745127
   -7.39766216  -1.48342931  -5.54817295  -1.55900335  -6.21558189
   -0.90341872  -1.09331369  -4.04764938  -6.02619839  -0.36431164
   -6.91541195  -0.30134696  -5.56458569  -0.06254807 -20.10161018
  -19.19358253 -19.20947266]
 [  0.          -5.74851036  -2.23256731  -6.48529243  -7.24711752
   -8.35337448  -0.77977628  -9.00911045  -5.5369463   -2.41623211
  -10.0587616   -2.51584196  -5.69090509  -6.00049019  -0.92609143
   -0.53638923  -2.71707201  -6.62812757  -4.05227852  -5.09376955
   -0.06831792 -19.68267441]]
